# COGS 108 - EDA Checkpoint

## Authors

Team list and Credits:

Mohan Dong: Project administration, Data curation, Conceptualization, Software, Writing - review & editing.

James Bartelloni: Methodology, Experimental investigation, Conceptualization, Writing - original draft.

Emily Vega: Conceptualization, Background Research, Writing - original draft, Writing - review & editing.

Nalin Joshi: Data curation, Software, Conceptualization, Writing - original draft.

Alexis Garcia: Methodology, Expiremental investigation, Writing = review & editing.

# Research Question

How did the acoustic characteristics of popular music change from the pre-COVID period (2017–2019) to the post-COVID period (2022), and are shifts in mainstream genre popularity associated with measurable changes in audio signal features such as spectral centroid (brightness), spectral rolloff (high-frequency energy), and zero crossing rate (percussiveness)?

## Background and Prior Work

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your proposal feedback

# Hypothesis


We hypothesize that the acoustic properties of popular music shifted following the COVID-19 pandemic. Specifically, we expect that songs released in the post-COVID period (2022) will exhibit higher spectral centroid, spectral rolloff, and zero crossing rate values compared to songs from the pre-COVID period (2017–2019).

## Data

### Data overview

Instructions: REPLACE the contents of this cell with your work, including any updates to recover points lost in your data checkpoint feedback


In [ ]:
# Run this code every time when you're actively developing modules in .py files.  It's not needed if you aren't making modules
#
## this code is necessary for making sure that any modules we load are updated here 
## when their source code .py files are modified

%load_ext autoreload
%autoreload 2

In [ ]:
# Setup code -- Run only once after cloning!!! 
#
# this code downloads the data from its source to the `data/00-raw/` directory
# if the data hasn't updated you don't need to do this again!

# if you don't already have these packages (you should!) uncomment this line
# %pip install requests tqdm

import sys
sys.path.append('./modules') # this tells python where to look for modules to import

import get_data # this is where we get the function we need to download data

# replace the urls and filenames in this list with your actual datafiles
# yes you can use Google drive share links or whatever
# format is a list of dictionaries; 
# each dict has keys of 
#   'url' where the resource is located
#   'filename' for the local filename where it will be stored 
datafiles = [
    { 'url': 'https://raw.githubusercontent.com/fivethirtyeight/data/refs/heads/master/airline-safety/airline-safety.csv', 'filename':'airline-safety.csv'},
    { 'url': 'https://raw.githubusercontent.com/fivethirtyeight/data/refs/heads/master/bad-drivers/bad-drivers.csv', 'filename':'bad-drivers.csv'}
]

get_data.get_raw(datafiles,destination_directory='data/00-raw/')

### Dataset #1: Billboard Top 50 Singles Data & Audio Features (1973-2022)

This dataset contains the Billboard Top-50 singles for each year from 1973 to 2022 along with nine audio signal features extracted directly from the waveform of each song. Because the dataset links chart popularity with measurable sound characteristics, it is well suited for studying how the acoustic character of mainstream music changes across time periods such as pre- and post-COVID.

Each observation corresponds to one song in the year-end chart and includes numerical descriptors derived from digital signal processing. Unlike subjective metadata like genre labels, these variables quantify the physical properties of sound.

The variables are standardized signal-analysis measures computed from amplitude and frequency distributions. They correspond closely to human perception of music:
- Chroma STFT (Short-Time Fourier Transform): compresses audio into 12 bins representing the Western chromatic scale (C, C#, D...), regardless of octave. It calculates the energy distribution of pitch classes over time and captures harmonic progression and melody.
- Chroma CENS (Chroma Energy Normalized Statistics): refines the result of standard Chroma STFT to make data robust, smoothed, and suitable for tasks like audio matching, synchronization, and structure analysis. Usually used in comparing musical passages even when they differ in performance style, instrumentation, etc.
- RMSE (Root Mean Square Energy) – Measures average loudness/intensity of the track. Higher values indicate louder, more energetic songs. For listeners this corresponds to musical “power” or punch.
- Spectral Centroid – Represents where most frequency energy is located (low vs high frequencies). Lower values sound darker or warmer, while higher values sound brighter and sharper.
- Spectral Rolloff – Frequency below which most sound energy lies. Higher rolloff suggests more treble/high-frequency content and is often associated with energetic or modern production.
- Spectral Bandwidth – Spread of frequencies present in the audio. Larger bandwidth indicates fuller, richer instrumentation.
- Zero Crossing Rate (ZCR) – Measures how frequently the waveform changes sign. Higher values indicate more percussive or noisy sounds (e.g., electronic or heavily rhythmic tracks).
- MFCC (Mel Frequency Cepstral Coefficients) – A compact representation of timbre (“sound color”), distinguishing acoustic vs electronic textures.
- Spectral Contrast – Difference between harmonic tones and noise components, related to clarity vs distortion.

In practical terms, increases in centroid, rolloff, bandwidth, ZCR, and RMSE generally correspond to music that sounds brighter, more energetic, richer, more dynamic, and louder; while decreases correspond to softer or more mellow sound characteristics. As we focuses on the mood or emotion-related acoustic character, we focus on Spectral Centroid (brightness), Spectral Rolloff (pitch-energy), and Zero Crossing Rate (dynamic and percussion) in this project as they best reflect the brightness and dynamics of a music.

Although the dataset contains many years of music, it only includes the Top-50 most popular songs each year. Therefore, it represents mainstream commercial music rather than all music produced during that period. Niche genres and underground artists are excluded, meaning conclusions should be interpreted as describing popular music trends rather than a universal musical trend.

Another concern is that audio features are derived algorithmically from recordings rather than listener perception. For example, a song may feel emotionally positive due to lyrics or cultural links, but the dataset only captures acoustic properties. Thus the dataset measures sound characteristics, not full emotional meaning.

Additionally, production technology evolves over decades: recording quality, mastering techniques, and compression may improve every year. Some feature differences may therefore reflect improvements in recording technology rather than cultural preference. This is particularly important when comparing modern post-COVID songs to older tracks.

Finally, the dataset ignores listening frequency, streaming counts, and regional variation. A highly streamed song and a barely-ranked #50 song are treated equally, which may reduce sensitivity when analyzing popularity intensity.

In [ ]:
# Download dataset
!pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kaavyamahajan/billboard-top-50-singles-data-features-1973-2022")

print("Path to dataset files:", path)

import shutil
from pathlib import Path

# Target directory (relative path)
target_dir = Path("data/00-raw")

# Copy dataset contents
shutil.copytree(path, target_dir, dirs_exist_ok=True)

print("Dataset copied to:", target_dir.resolve())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
feature_df = pd.read_csv('data/00-raw/top_50_song_features.csv')
chart_df = pd.read_csv('data/00-raw/top_50s_chart.csv')

# Rename the first column to 'Year' and delete the first 3 rows
feature_df.columns.values[0] = 'Year'
feature_df = feature_df.iloc[3:].reset_index(drop=True)

feature_df.head()

### Rename the first column to 'Year' and delete the first 3 rows

In [ ]:

feature_df.columns.values[0] = 'Year'
feature_df = feature_df.iloc[3:].reset_index(drop=True)
feature_df.head()

### For the purposes of the research question keep only rows where 2016 <= Year <= 2019 for pre_covid

In [ ]:

pre_COVID_feature_df = feature_df[feature_df['Year'].isin(range(2017, 2019))]
pre_COVID_feature_df.head()


### For the purposes of the research question keep only rows where = Year = 2022 for post_covid

In [ ]:
post_COVID_feature_df = feature_df[feature_df['Year'].isin(range(2022, 2024))]
post_COVID_feature_df.head()

### Dataset #2
 as above, add any more copies of this that you need to given how many datasets you have

In [ ]:
## YOUR CODE TO LOAD/CLEAN/TIDY/WRANGLE THE DATA GOES HERE
## FEEL FREE TO ADD MULTIPLE CELLS PER SECTION 

## Results

### Exploratory Data Analysis

Instructions: replace the words in this subsection with whatever words you need to setup and preview the EDA you're going to do.   

Please explicitly load the fully wrangled data you will use from `data/02-processed`.  This is a good idea rather than forcing people to re-run the data getting / wrangling cells above.  Sometimes it takes a long time to get / wrangle data compared to reloading the fixed up dataset.

Carry out whatever EDA you need to for your project in the code cells below.  Because every project will be different we can't really give you much of a template at this point. But please make sure you describe the what and why in text here as well as providing interpretation of results and context.

Please note that you should consider the use of python modules in your work.  Any code which gets called repeatedly should be modularized. So if you run the same pre-processing, analysis or visualiazation on different subsets of the data, then you should turn that into a function or class.  Put that function or class in a .py file that lives in `modules/`.  Import the module you made and use it to get your work done.  For reference see `get_raw()` which is inside `modules/get_data.py`. 



### Pre-COVID Music Feature Analysis

This section performs comprehensive exploratory data analysis on the pre-COVID music feature dataset (2017-2019), focusing on spectral audio features that capture the acoustic characteristics of music tracks.

In [ ]:
#### Step 1: Data Loading and Cleaning

In [ ]:
# Step 1: Data Loading and Cleaning
df = pre_COVID_feature_df.copy()
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

In [ ]:
# Check missing values
missing_pct = (df.isnull().sum() / len(df)) * 100
print("Missing value percentages:")
print(missing_pct[missing_pct > 0].sort_values(ascending=False).head(20))

In [ ]:
# Drop columns with more than 40% missing values
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
if cols_to_drop:
    print(f"Dropping {len(cols_to_drop)} columns")
    df = df.drop(columns=cols_to_drop)

In [ ]:
# Fill remaining missing values with mean
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

#### Step 2: Basic Dataset Exploration

In [ ]:
spectral_rolloff_features = [col for col in df.columns if 'spectral_rolloff' in col.lower()]
print(f"Spectral Rolloff features: {len(spectral_rolloff_features)}")

In [ ]:
# Summary statistics for spectral_rolloff features
# Convert to numeric and get mean, std, min, max, etc.
rolloff_df = df[spectral_rolloff_features].apply(pd.to_numeric, errors='coerce')
print("Summary Statistics for Spectral Rolloff Features:")
print(rolloff_df.describe())



In [ ]:
zcr_features = [col for col in df.columns if 'zcr' in col.lower()]
print(f"ZCR features: {len(zcr_features)}")

In [ ]:
# Summary statistics for zcr features
# Convert to numeric and get mean, std, min, max, etc.
zcr_df = df[zcr_features].apply(pd.to_numeric, errors='coerce')
print("Summary Statistics for ZCR Features:")
print(zcr_df.describe())

In [ ]:
print(df[zcr_features].describe())

#### Step 3: Focus on Audio Spectral Features

In [ ]:
# Summary statistics for spectral_centroid features
# Convert to numeric and get mean, std, min, max, etc.
centroid_df = df[spectral_centroid_features].apply(pd.to_numeric, errors='coerce')
print("Summary Statistics for Spectral Centroid Features:")
print(centroid_df.describe())

In [ ]:
# Step 3: Focus on Audio Spectral Features
spectral_centroid_features = [col for col in df.columns if 'spectral_centroid' in col.lower()]
print(f"Spectral Centroid features: {len(spectral_centroid_features)}")

In [ ]:
print(df[spectral_centroid_features].describe())

#### Step 4: Visualization of Key Audio Features

**Audio Feature Explanations:**
- **Spectral Centroid**: Represents the "brightness" of sound - higher values indicate brighter, sharper sounds (more high-frequency content)
- **Spectral Rolloff**: Measures high-frequency energy distribution - indicates where most of the spectral energy is concentrated
- **Zero Crossing Rate (ZCR)**: Measures noisiness/percussiveness - higher values indicate more percussive or noisy sounds (e.g., electronic or heavily rhythmic tracks)

In [ ]:
# Step 4: Visualization of Key Audio Features
# Distribution plots for spectral_centroid
if len(spectral_centroid_features) > 0:
    n_cols = min(4, len(spectral_centroid_features))
    n_rows = min(3, (len(spectral_centroid_features) + n_cols - 1) // n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten()
    
    for i, col in enumerate(spectral_centroid_features[:n_rows*n_cols]):
        df[col].hist(bins=30, ax=axes[i], alpha=0.7)
        axes[i].set_title(col)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution plots for spectral_rolloff
if len(spectral_rolloff_features) > 0:
    n_cols = min(4, len(spectral_rolloff_features))
    n_rows = min(3, (len(spectral_rolloff_features) + n_cols - 1) // n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten()
    
    for i, col in enumerate(spectral_rolloff_features[:n_rows*n_cols]):
        df[col].hist(bins=30, ax=axes[i], alpha=0.7)
        axes[i].set_title(col)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution plots for zcr
if len(zcr_features) > 0:
    n_cols = min(4, len(zcr_features))
    n_rows = min(3, (len(zcr_features) + n_cols - 1) // n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten()
    
    for i, col in enumerate(zcr_features[:n_rows*n_cols]):
        df[col].hist(bins=30, ax=axes[i], alpha=0.7)
        axes[i].set_title(col)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap among all selected spectral features
all_spectral_features = spectral_centroid_features + spectral_rolloff_features + zcr_features

if len(all_spectral_features) > 0:
    
    spectral_corr = df[all_spectral_features].corr()
    
    plt.figure(figsize=(14, 12))
    ax = sns.heatmap(
        spectral_corr, 
        annot=False,  # Too many features to annotate
        cmap='coolwarm', 
        center=0,
        square=True,
        linewidths=0.3,
        cbar_kws={"shrink": 0.8},
        xticklabels=False,  # Hide labels for readability
        yticklabels=False
    )
    plt.title(
        'Correlation Heatmap: All Spectral Features\n(Spectral Centroid, Rolloff, and ZCR)', 
        fontsize=14, fontweight='bold', pad=20
    )
    plt.xlabel("Spectral Features")
    plt.ylabel("Spectral Features")
    plt.tight_layout()
    plt.show()
    
    print(f"Correlation matrix shape: {spectral_corr.shape}")
    print(f"Features included: {len(all_spectral_features)}")
else:
    print("No spectral features found for correlation analysis")

#### Section 2 of EDA if you need it  - please give it a better title than this

Some more words and stuff.  Remember notebooks work best if you interleave the code that generates a result with properly annotate figures and text that puts these results into context.

In [ ]:
## YOUR CODE HERE
## FEEL FREE TO ADD MULTIPLE CELLS PER SECTION

## Ethics

**A. Data Collection**

[X] A.1 Informed consent This project uses existing data that is publicly available or anonymized. We are not collecting new data from people. Any consent was handled by the original data sources.

[X] A.2 Collection bias The data may not represent all groups equally. Some groups may be overrepresented while others are missing. We will avoid overgeneralizing results and will discuss these limits in our analysis.

[X] A.3 Limit PII exposure The datasets do not include personally identifiable information. We will not try to identify individuals or use sensitive information that is not needed for the analysis.

[X] A.4 Downstream bias mitigation If differences across groups appear, we will treat them as correlations rather than causes. We will discuss how bias in the data could influence these patterns.

**B. Data Storage**

[X] B.1 Data security All data will be stored in the UCSD DataHub environment and managed through GitHub. Access is limited to group members and the data will not be shared outside of the course.

[X] B.2 Right to be forgotten Because this project uses anonymized secondary data, individual record removal is not possible. We will follow any rules set by the data providers.

[X] B.3 Data retention plan The data will only be used for this course project. It will not be kept or reused after the course ends.

**C. Analysis**

[X] C.1 Missing perspectives Our interpretations may reflect our own viewpoints. To reduce this, we will rely on course concepts and existing research when explaining results.

[X] C.2 Dataset bias We will check for issues like missing data and class imbalance. If these issues affect results, we will clearly explain them.

[X] C.3 Honest representation All visualizations and summaries will accurately reflect the data. We will avoid misleading visuals or overstating findings.

[X] C.4 Privacy in analysis The data does not include personal information. We will not introduce or infer private details during analysis.

[X] C.5 Auditability Our analysis process will be documented with clear code and version control so results can be reproduced.

**D. Modeling**

[X] D.1 Proxy discrimination We will be careful with variables that could act as proxies for sensitive traits. Any concerns will be discussed in the analysis.

[X] D.2 Fairness across groups If group comparisons are used, we will check for unequal outcomes and note them in the discussion.

[X] D.3 Metric selection We will choose metrics that fit the research question and explain why they were selected.

[X] D.4 Explainability Model choices and results will be explained in plain language.

[X] D.5 Communicate limitations We will clearly describe the limits of our models and data so results are not misinterpreted.

**E. Deployment**

[X] E.1 Monitoring and evaluation This project is for a class and will not be deployed. We will still evaluate results carefully and discuss limitations.

[X] E.2 Redress If issues are identified, we will revise the analysis and document changes.

[X] E.3 Roll back If a method or model causes problems, it can be removed and updated.

[X] E.4 Unintended use Results will be presented with enough context to reduce misuse or misunderstanding.

## Team Expectations 


### Team Expectation 1: Communication and Meetings

The team will primarily use **Discord** as its main platform for communication. Discord has already proven effective because all members regularly check it and typically respond at least once per day, with increased activity as deadlines or checkpoints approach. This platform will be used for general updates, task coordination, and quick clarifications. If a team member is non-responsive on Discord and the matter is time-sensitive, **email will serve as a backup communication channel**.
The team will meet **at least once per week**, with the option to increase meetings to twice per week during periods of heavier workload or approaching milestones. Meetings may take place **in person or virtually**, depending on availability. Consistent attendance and communication are expected from all members to maintain steady progress on the project.

---

### Team Expectation 2: Tone and Interpersonal Conduct

During the first in-person meeting, the team will establish communication norms and expectations for interpersonal conduct. The agreed-upon tone will be **respectful, direct, and constructive**. Direct feedback is encouraged because it helps improve the project, but communication must always remain professional and respectful.
Personal insults, dismissive language, or hostile behavior will not be tolerated. Team members should feel comfortable expressing concerns, asking questions, or disagreeing with ideas without fear of judgment. The team values **honesty, clarity, and mutual respect**, and maintaining a supportive environment is considered a shared responsibility.

---

### Team Expectation 3: Decision-Making Process

The team will aim to make **major project decisions through group consensus whenever possible**. Members will openly discuss ideas and concerns to ensure that everyone’s perspective is considered. If disagreements arise, the team will work collaboratively to reach a compromise or adjust the proposed approach.
For smaller tasks or specialized work, **decision-making authority may be delegated to the team member responsible for that component**. Those members are trusted to make informed decisions while still keeping the rest of the team updated. If an urgent decision is required and not all members are available, the members present will make a decision that best supports the progress of the project.

---

### Team Expectation 4: Task Allocation and Role Structure

Specific roles will be discussed and finalized during the team’s first in-person meeting. Until roles are formally assigned, leadership has naturally emerged with **Mohan coordinating tasks while all members contribute equally**.
Tasks will generally be assigned based on **individual interest and strengths**, while ensuring that responsibilities remain balanced across the group. No team member should feel overwhelmed or solely responsible for large portions of the work. Progress and task assignments will be tracked through **GitHub**, allowing all members to stay informed about the status of the project.

---

### Team Expectation 5: Support and Handling Challenges

If a team member encounters difficulties with a task, they are expected to **communicate the issue early** so the team can provide support. The team recognizes that members have different strengths, particularly in data science and programming, and will use those strengths to help each other when necessary.
Workload adjustments may be made when needed so that no individual consistently carries an excessive portion of the work. A collaborative and supportive approach will help ensure steady progress while maintaining fairness among team members.

---

### Team Expectation 6: Planning, Workflow, and Deadlines

The team will follow an **agile-style workflow**, organizing work into short development cycles or "sprints" lasting approximately two to three weeks. Tasks, goals, and deadlines will be reviewed regularly and adjusted as the project progresses.
While the team values flexibility in adapting plans, deadlines will still be treated seriously. Members are expected to complete assigned tasks on time and communicate proactively if delays occur. The team may use tools such as **GitHub or Jira** to track progress and organize tasks throughout the project.

---

### Team Expectation 7: Documentation and Visibility of Expectations

All team expectations, guidelines, and communication policies will be documented in a dedicated **#rules channel on the team’s Discord server**. Important messages outlining team expectations will be pinned so they remain easily accessible to all members.
Maintaining clear documentation helps ensure transparency, reduces misunderstandings, and keeps all team members aligned throughout the project. If expectations change during the project, updates will be discussed and documented so everyone remains informed.

## Project Timeline Proposal

| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 1/28  |  9 PM | Reviewed the Google forum and briefly looked through several past projects to identify ones that stood out. Considered possible topic areas that might interest the group. | Shared project ideas and discussed which topics seemed most interesting and feasible. Decided to use a Google Form to give everyone time to explore project options before making a final decision. |
| 1/30  |  6 PM | Reviewed the requirements for the project review and revisited the most promising past projects for inspiration. | Worked together on the project review. Discussed the two projects that inspired us the most and how they could help shape our own project direction. |
| 2/4   |  2 PM | Reviewed the project proposal guidelines and brainstormed potential research questions. | Chose our project topic (Music during the COVID pandemic). Drafted and refined our hypothesis until we reached one that the group agreed on. Split up the proposal writing tasks and began drafting our sections. |
| 2/10 | Online | Began searching for datasets related to music popularity, genres, and audio features during different time periods. | Shared potential datasets and evaluated which ones would best help answer our research question. Discussed potential limitations or missing variables in each dataset. |
| 2/18 | Online 6 PM | Reviewed the datasets more closely and took notes on useful variables and structure. | Finalized the datasets we would use for the project. Split up tasks for cleaning and exploring the datasets. Ensured everyone understood their responsibilities and the goals for the next checkpoint. |
| 2/23 | Online 3 PM| Explored methods for cleaning the data and resolving errors encountered when loading or analyzing the datasets. | Shared data cleaning approaches and discussed how to standardize the datasets. Began examining possible changes in genre popularity and audio features across years. |
| 3/4 | Online 1 PM| Ran the main analysis comparing genre popularity and audio features between pre-COVID and post-COVID periods. | Discussed initial results and whether they supported our hypothesis. Talked about visualization ideas and how to clearly present the analysis in the final project. Gathered new data and updated the research proposal and hypothesis. |
| 3/13 | Online/Geisel 5 PM| Drafted the results, discussion, and conclusion sections of the project report. | Reviewed the entire project together and suggested edits or improvements. Began planning the structure and narrative flow of the final presentation video. |
| 3/14 | Online/Geisel 11 AM | Prepared an outline and script ideas for the final project video. | Began recording and editing the project video. Ensured the explanation of the analysis and results was clear and concise. |
| 3/18 | Online/Geisel 12 PM| Individually reviewed the project notebooks and video for any remaining issues or missing components. | Conducted a final group review of the entire project. Checked that all branches were properly merged, ensured the video was finalized, and confirmed everything was ready for submission. |